<a href="https://colab.research.google.com/github/docling-project/docling/blob/main/docs/examples/xbrl_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XBRL Document Conversion

This example demonstrates how to parse XBRL (eXtensible Business Reporting Language) documents using Docling, completely offline.

XBRL is a standard XML-based format used globally by companies, regulators, and financial institutions for exchanging business and financial information in a structured, machine-readable format. It's widely adopted for regulatory filings (e.g., SEC filings in the US).

## What you'll learn

- How to configure Docling to parse XBRL documents offline
- How to provide a local taxonomy package for XBRL validation
- How to extract structured data from XBRL instance documents
- How to export XBRL content to various formats (Markdown, JSON, etc.)
- How to parse inline XBRL (iXBRL) documents from SEC's EDGAR, with a local taxonomy, from a URL, or with remote taxonomy fetching

The data to run this notebook has been fetched from the [SEC's Electronic Data Gathering, Analysis, and Retrieval (EDGAR)](https://www.sec.gov/search-filings) system.

## Setup

Install Docling with XBRL support (the `[xbrl]` extra pulls in the `arelle-release` dependency):

In [1]:
%pip install -q 'docling[xbrl]'

## Download Sample XBRL Data

For this example, we'll use a sample XBRL instance document and its taxonomy. In a real scenario, you would have your own XBRL files and taxonomy packages.

We'll download the test data from the Docling repository:

In [1]:
import urllib.request
from pathlib import Path

# Create directories for XBRL data
data_dir = Path("xbrl_data")
taxonomy_dir = data_dir / "taxonomy"
taxonomy_dir.mkdir(parents=True, exist_ok=True)

# Base URL for test data
base_url = "https://raw.githubusercontent.com/docling-project/docling/main/tests/data/xbrl/sources/"

# Download XBRL instance file
instance_file = data_dir / "mlac-20251231.xml"
if not instance_file.exists():
    print("Downloading XBRL instance file...")
    urllib.request.urlretrieve(f"{base_url}mlac-20251231.xml", instance_file)
    print(f"Downloaded: {instance_file}")

# Download taxonomy files
taxonomy_files = [
    "mlac-20251231.xsd",
    "mlac-20251231_cal.xml",
    "mlac-20251231_def.xml",
    "mlac-20251231_lab.xml",
    "mlac-20251231_pre.xml",
]

print("Downloading taxonomy files...")
for filename in taxonomy_files:
    target_file = taxonomy_dir / filename
    if not target_file.exists():
        urllib.request.urlretrieve(f"{base_url}mlac-taxonomy/{filename}", target_file)
        print(f"  Downloaded: {filename}")

# Download taxonomy package (contains URL mappings for offline parsing)
taxonomy_package = taxonomy_dir / "taxonomy_package.zip"
if not taxonomy_package.exists():
    print("Downloading taxonomy package...")
    urllib.request.urlretrieve(
        f"{base_url}mlac-taxonomy/taxonomy_package.zip", taxonomy_package
    )
    print("  Downloaded: taxonomy_package.zip")

print("\nAll files downloaded successfully!")


All files downloaded successfully!


## Configure XBRL Backend

To parse XBRL documents offline, we need to:

1. Enable local resource fetching (for taxonomy files)
2. Disable remote resource fetching (for offline operation)
3. Provide the path to the local taxonomy directory

In [2]:
from docling.datamodel.backend_options import XBRLBackendOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, XBRLFormatOption

# Configure XBRL backend options
backend_options = XBRLBackendOptions(
    enable_local_fetch=True,  # Allow reading local taxonomy files
    enable_remote_fetch=False,  # Disable remote fetching for offline operation
    taxonomy=taxonomy_dir,  # Path to local taxonomy directory
)

# Create document converter with XBRL support
converter = DocumentConverter(
    allowed_formats=[InputFormat.XML_XBRL],
    format_options={
        InputFormat.XML_XBRL: XBRLFormatOption(backend_options=backend_options)
    },
)

print("XBRL converter configured successfully!")

XBRL converter configured successfully!


💡 Because the converter must read the supporting taxonomy files, set the `enable_local_fetch` option to **True** in the XBRL backend settings.  
💡 In addition to the XBRL report's own taxonomy files, you need a *taxonomy package*-a bundle containing URL remappings that enables completely offline parsing. If you prefer not to supply a taxonomy package, omit it and set `enable_remote_fetch` to **True** in the XBRL backend settings. The backend will fetch the web‑referenced files from the remote publishers and cache them locally for reuse.

## Convert XBRL Document

Now we can convert the XBRL instance document. The converter will:

- Parse the XBRL instance file
- Validate it against the local taxonomy
- Extract metadata, text blocks, and numeric facts
- Convert everything to a unified DoclingDocument representation

In [3]:
# Convert the XBRL document
print(f"Converting XBRL document: {instance_file}")
result = converter.convert(instance_file)
doc = result.document

print("\nConversion successful!")
print(f"Document name: {doc.name}")
print(f"Number of items: {len(list(doc.iterate_items()))}")

Converting XBRL document: xbrl_data/mlac-20251231.xml

Conversion successful!
Document name: mlac-20251231
Number of items: 236


## Inspect Document Structure

Let's examine the structure of the converted document:

In [4]:
from docling_core.types.doc import DocItemLabel

# Count items by type
item_counts = {}
for item, _ in doc.iterate_items():
    label = item.label
    item_counts[label] = item_counts.get(label, 0) + 1

print("Document structure:")
for label, count in sorted(item_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label.value}: {count}")

Document structure:
  text: 211
  table: 23
  title: 1
  key_value_region: 1


## View Sample Content

Let's look at some of the extracted content:

In [5]:
# Display first few text items
print("Sample text content:\n")
text_count = 0
for item, _ in doc.iterate_items():
    if item.label == DocItemLabel.TEXT and text_count < 3:
        print(f"- {item.text[:200]}..." if len(item.text) > 200 else f"- {item.text}")
        print()
        text_count += 1

Sample text content:

- None

- We are a special purpose acquisition company with no business operations. Since our initial public offering, our sole business activity has been identifying and evaluating suitable acquisition transac...

- We depend on digital technologies, including information systems, infrastructure and cloud applications and services, including those of third parties with which we may deal. Sophisticated and deliber...



## View Key-Value Pairs

XBRL numeric facts are extracted as key-value pairs:

In [6]:
# Display sample key-value pairs
graph_data = doc.key_value_items[0].graph
print(f"Total key-value pairs extracted: {len(graph_data.links)}\n")
for link in graph_data.links[:10]:
    source = next(
        item for item in graph_data.cells if item.cell_id == link.source_cell_id
    )
    target = next(
        item for item in graph_data.cells if item.cell_id == link.target_cell_id
    )
    print(f"{source.text} -> {target.text}")

Total key-value pairs extracted: 1842

EntityPublicFloat -> value: 239160600
EntityPublicFloat -> period: 2025-06-30
EntityPublicFloat -> currency: USD
EntityPublicFloat -> decimals: 0
EntityCommonStockSharesOutstanding -> value: 23805000
EntityCommonStockSharesOutstanding -> period: 2026-02-20
EntityCommonStockSharesOutstanding -> currency: shares
EntityCommonStockSharesOutstanding -> decimals: INF
EntityCommonStockSharesOutstanding -> value: 7187500
EntityCommonStockSharesOutstanding -> period: 2026-02-20


💡 The current backend implementation flattens all key‑value pairs in an XBRL report. Future improvements will preserve the rich taxonomy of those data points.

## Export to Markdown

Export the document to Markdown format for easy reading:

In [7]:
# Export to Markdown
markdown_content = doc.export_to_markdown()

# Display first 2000 characters
print("Markdown export (first 2000 characters):\n")
print(markdown_content[:2000])
print("\n...")

# Save to file
output_md = data_dir / "output.md"
output_md.write_text(markdown_content)
print(f"\nFull markdown saved to: {output_md}")

Markdown export (first 2000 characters):

# 10-K MOUNTAIN LAKE ACQUISITION CORP. 2025-12-31

None

We are a special purpose acquisition company with no business operations. Since our initial public offering, our sole business activity has been identifying and evaluating suitable acquisition transaction candidates. Therefore, we do not consider that we face significant cybersecurity risk and have not adopted any cybersecurity risk management program or formal processes for assessing cybersecurity risk.

We depend on digital technologies, including information systems, infrastructure and cloud applications and services, including those of third parties with which we may deal. Sophisticated and deliberate attacks on, or security breaches in, our information systems or infrastructure, or the information systems or infrastructure of third parties or the cloud, could lead to corruption or misappropriation of our assets, proprietary information and sensitive or confidential data. Because of o

## Export to DocLang

Export the complete document structure to [DocLang](https://doclang.ai/), the new AI-native document format:

In [8]:
import json

# Export to JSON
output_dclg = data_dir / "output.dclg"
doc.save_as_doclang(output_dclg)

print(f"Document exported to JSON: {output_dclg}")
print(f"File size: {output_dclg.stat().st_size / 1024:.2f} KB")

Document exported to JSON: xbrl_data/output.dclg
File size: 105.49 KB


You can visualize the exported document with the online [DocgLang Viewer](https://doclang.ai/viewer/) tool.

---

## Inline XBRL (iXBRL) — SEC EDGAR Reports

### What is iXBRL?

**Inline XBRL (iXBRL)** embeds structured XBRL facts directly inside a human-readable HTML document using the `ix:` namespace. Unlike a traditional XBRL instance file (`.xml`) that contains only machine-readable data, an iXBRL document (`.htm` / `.html`) is simultaneously a readable financial report and a structured data source — the same file serves both purposes.

iXBRL is now the **mandatory submission format** for:
- **SEC EDGAR** structured-data disclosures (10-K, 10-Q, 8-K, …)
- **UK Companies House** filings
- **European Single Electronic Format (ESEF)** filings

Docling detects iXBRL documents automatically, so no special configuration is needed to tell them apart from plain HTML. As with XBRL instance files, the iXBRL document is passed to a `DocumentConverter` together with the options in `XBRLBackendOptions`.

### SEC EDGAR as a source

The [SEC EDGAR full-text search](https://efts.sec.gov/LATEST/search-index?q=%22inline+xbrl%22&dateRange=custom&startdt=2024-01-01&enddt=2025-12-31&forms=10-K) platform lets you browse and download structured filings. For example, IBM's Q3 2026 10-Q report is available as:

- **Human-readable iXBRL report:** https://www.sec.gov/Archives/edgar/data/51143/000005114326000095/ibm-20260930.htm
- **Filing index (all related files):** https://www.sec.gov/Archives/edgar/data/51143/000005114326000095/index.json

Besides the iXBRL report, the filing includes the company-specific *extension taxonomy* files (`ibm-20260930.xsd` and its `_cal.xml`, `_def.xml`, `_lab.xml` and `_pre.xml` linkbases). The standard base taxonomies (US-GAAP, SRT, DEI, …) are only referenced by URL.

## Download Sample iXBRL Data

We download the IBM 10-Q report and its extension taxonomy files from the Docling repository. We also fetch a *taxonomy package* (`taxonomy_package.zip`) that bundles the standard base taxonomies with URL remappings, so the report can be parsed completely offline.

In [10]:
# Directories for the iXBRL data
ixbrl_dir = data_dir / "ixbrl"
ixbrl_taxonomy_dir = ixbrl_dir / "ibm-taxonomy"
ixbrl_taxonomy_dir.mkdir(parents=True, exist_ok=True)

ixbrl_base_url = "https://raw.githubusercontent.com/docling-project/docling/main/tests/data/xbrl/ixbrl/"

# Download the iXBRL report
ixbrl_file = ixbrl_dir / "ibm-20260930.htm"
if not ixbrl_file.exists():
    print("Downloading iXBRL report...")
    urllib.request.urlretrieve(f"{ixbrl_base_url}ibm-20260930.htm", ixbrl_file)
    print(f"Downloaded: {ixbrl_file}")

# Download the extension taxonomy files and the taxonomy package
ixbrl_taxonomy_files = [
    "ibm-20260930.xsd",
    "ibm-20260930_cal.xml",
    "ibm-20260930_def.xml",
    "ibm-20260930_lab.xml",
    "ibm-20260930_pre.xml",
    "taxonomy_package.zip",
]

print("Downloading taxonomy files...")
for filename in ixbrl_taxonomy_files:
    target_file = ixbrl_taxonomy_dir / filename
    if not target_file.exists():
        urllib.request.urlretrieve(
            f"{ixbrl_base_url}ibm-taxonomy/{filename}", target_file
        )
        print(f"  Downloaded: {filename}")

print("\nAll files downloaded successfully!")

Downloaded: xbrl_data/ixbrl/ibm-20260930.htm
  Downloaded: ibm-20260930.xsd
  Downloaded: ibm-20260930_cal.xml
  Downloaded: ibm-20260930_def.xml
  Downloaded: ibm-20260930_lab.xml
  Downloaded: ibm-20260930_pre.xml
  Downloaded: taxonomy_package.zip

All files downloaded successfully!


## Parse an iXBRL Report with a Local Taxonomy (Offline)

Parsing an iXBRL report works in the same way as parsing an XBRL instance file. We point the `taxonomy` option to a folder that contains:

- the extension taxonomy files of the report (`.xsd` schema and linkbases), and
- optionally, a *taxonomy package* (`.zip`) with the standard base taxonomies and the URL remappings needed for offline parsing.

With both, the report is parsed without any network access.

💡 If the taxonomy package maps every `https://` URL to its own files (a catch-all `catalog.xml`, as in the sample package), it must also contain the extension taxonomy of the report, or the extension files must be in the `taxonomy` folder next to it. Docling does not fetch from the web what the package maps, so a missing extension taxonomy leaves the facts unresolved. In that case, the backend logs a warning with the number of facts that could not be linked to a concept.

In [10]:
# Same backend options as for XBRL instance files: the iXBRL report only
# needs a different input file.
ixbrl_backend_options = XBRLBackendOptions(
    enable_local_fetch=True,  # Read the local taxonomy files
    enable_remote_fetch=False,  # Offline operation
    taxonomy=ixbrl_taxonomy_dir,  # Folder with extension taxonomy + taxonomy package
)

ixbrl_converter = DocumentConverter(
    allowed_formats=[InputFormat.XML_XBRL],
    format_options={
        InputFormat.XML_XBRL: XBRLFormatOption(backend_options=ixbrl_backend_options)
    },
)

print(f"Converting iXBRL document: {ixbrl_file}")
ixbrl_result = ixbrl_converter.convert(ixbrl_file)
ixbrl_doc = ixbrl_result.document

print(f"Document name : {ixbrl_doc.name}")
print(f"Number of items: {len(list(ixbrl_doc.iterate_items()))}")
print(f"Text items: {len(ixbrl_doc.texts)}")
print(f"Tables: {len(ixbrl_doc.tables)}")
print(f"Key-value pairs: {len(ixbrl_doc.key_value_items[0].graph.links)}")

Converting iXBRL document: /Users/ceb/git/docling-3/tests/data/xbrl/ixbrl/ibm-20260930.htm
Document name : ibm-20260930
Number of items: 920
Text items: 753
Tables: 166
Key-value pairs: 11199


In [11]:
# Preview the extracted content
ixbrl_md = ixbrl_doc.export_to_markdown(compact_tables=True)
print("Markdown preview (first 2000 characters):\n")
print(ixbrl_md[:2000])
print("\n...")

Markdown preview (first 2000 characters):

# 10-Q INTERNATIONAL BUSINESS MACHINES CORPORATION 2026-09-30

Basis of Presentation:

The accompanying Consolidated Financial Statements and footnotes of the International Business Machines Corporation (IBM or the company) have been prepared in accordance with accounting principles generally accepted in the United States of America (GAAP). The financial statements and footnotes are unaudited. In the opinion of the company's management, these statements include all adjustments, which are only of a normal recurring nature, necessary to present a fair statement of the company's results of operations, financial position and cash flows.

The preparation of financial statements in conformity with GAAP requires management to make estimates and assumptions that affect the amount of assets, liabilities, revenue, costs, expenses and other comprehensive income/(loss) that are reported in the Consolidated Financial Statements and accompanying disclosures

## Parse an iXBRL Report from a URL without Any Taxonomy File

The most compact option is to give Docling the URL of the report and set `enable_remote_fetch` to **True**. When the input is a URL, Docling remembers where the report was published. The backend resolves the relative references of the report (for instance, `ibm-20260930.xsd`, the company-specific extension taxonomy) against that URL. The referenced files, together with the standard base taxonomies (US-GAAP, SRT, DEI, …), are fetched from their publishers and cached locally for reuse.

No taxonomy file is needed. The first run requires network access and takes longer.

💡 **SEC EDGAR access policy**: EDGAR requires every HTTP request to carry a descriptive `User-Agent` header that identifies the application and includes a contact e-mail address (see https://www.sec.gov/developer). Pass it with the `headers` argument of `DocumentConverter.convert()`. Replace the placeholder below with your own application name and e-mail.

In [12]:
remote_backend_options = XBRLBackendOptions(
    enable_remote_fetch=True,  # Fetch the referenced taxonomies from the web
)

remote_converter = DocumentConverter(
    allowed_formats=[InputFormat.XML_XBRL],
    format_options={
        InputFormat.XML_XBRL: XBRLFormatOption(backend_options=remote_backend_options)
    },
)

ixbrl_url = (
    "https://www.sec.gov/Archives/edgar/data/51143/000005114326000095/ibm-20260930.htm"
)
sec_headers = {"User-Agent": "MyApp/1.0 your-email@example.com"}

print(f"Converting iXBRL document from URL: {ixbrl_url}")
remote_result = remote_converter.convert(ixbrl_url, headers=sec_headers)
remote_doc = remote_result.document

print(f"Number of items: {len(list(remote_doc.iterate_items()))}")
print(f"Text items: {len(remote_doc.texts)}")
print(f"Tables: {len(remote_doc.tables)}")

Converting iXBRL document from URL: https://www.sec.gov/Archives/edgar/data/51143/000005114326000095/ibm-20260930.htm
Number of items: 920
Text items: 753
Tables: 166


## Parse a Local iXBRL File with Remote Taxonomy Fetching

A local file or a stream has no origin that Docling can use to find the extension taxonomy. SEC reports reference it with a *relative* path, so setting only `enable_remote_fetch=True` is not enough: the facts cannot be linked to their concepts and only the title of the report is extracted.

You have two options:

- Provide the extension taxonomy files in the `taxonomy` folder (without a taxonomy package). Docling fetches only the standard base taxonomies from the web.
- Tell Docling where the report was published with the `source_uri` option. The relative references are then resolved against that URL, as in the previous example. Files found in the `taxonomy` folder always take precedence.

Here we use the second option with the local copy of the report:

In [13]:
source_uri_options = XBRLBackendOptions(
    enable_remote_fetch=True,
    source_uri=ixbrl_url,  # Where the local file was published
)

source_uri_converter = DocumentConverter(
    allowed_formats=[InputFormat.XML_XBRL],
    format_options={
        InputFormat.XML_XBRL: XBRLFormatOption(backend_options=source_uri_options)
    },
)

source_uri_doc = source_uri_converter.convert(ixbrl_file, headers=sec_headers).document
print(f"Number of items: {len(list(source_uri_doc.iterate_items()))}")

Number of items: 920


💡 **Which approach to choose?** For reproducible, offline processing, use a `taxonomy` folder with a taxonomy package. If network access is acceptable, set `enable_remote_fetch=True` and pass the URL of the report, or set `source_uri` for local files and streams. The same options work for XBRL instance files (`.xml`, `.xbrl`). Files in the `taxonomy` folder always take precedence over remote ones, so combining a taxonomy package with `enable_remote_fetch=True` fetches only what the package does not map.

---

## Summary

In this example, we demonstrated:

✅ How to configure Docling for offline XBRL parsing  
✅ How to provide a local taxonomy for XBRL validation  
✅ How to convert XBRL instance documents to DoclingDocument  
✅ How to extract metadata, text blocks, and numeric facts  
✅ How to export XBRL content to Markdown and JSON formats  
✅ How to parse iXBRL reports from SEC EDGAR with a local taxonomy (offline), from a URL, or with remote taxonomy fetching  

### Key Points

- **Offline operation**: By setting `enable_remote_fetch=False`, all processing happens locally
- **Taxonomy support**: The local taxonomy directory should contain all necessary schema and linkbase files
- **iXBRL auto-detection**: Docling detects inline XBRL documents by content-sniffing; no format flag is needed
- **Same options for XBRL and iXBRL**: Both use `XBRLBackendOptions` and `XBRLFormatOption`; only the input file changes
- **Taxonomy folder for iXBRL**: Put the extension taxonomy files and, optionally, a taxonomy package (`.zip`) in the `taxonomy` folder
- **Remote taxonomy resolution**: Set `enable_remote_fetch=True` to fetch the standard base taxonomies (US-GAAP, DEI) from the web instead of a taxonomy package
- **Parsing from a URL**: With `enable_remote_fetch=True`, the URL of the report is enough. Docling resolves the relative references to the extension taxonomy against that URL
- **Local files and streams**: Provide the extension taxonomy in the `taxonomy` folder or set `source_uri` to the URL where the report was published. Otherwise, only the title of an SEC report is extracted, because the extension schema is referenced with a relative path
- **Structured extraction**: XBRL numeric facts are extracted as key-value pairs with graph representation
- **Text blocks**: HTML text blocks in XBRL are converted to structured content

### Note on Future Changes

⚠️ The current implementation uses `DoclingDocument`'s `GraphData` object to represent key-value pairs. This design will change in a future release of the `docling-core` library.